LangExtract を使った設定抽出テスト (公開用)
======================================================
このノートは小説本文からキャラクター設定・舞台設定を抽出し、
LangExtract によるインタラクティブな HTML ビジュアライゼーションを生成するテストテンプレートです。  
日本語テキストだとchar_intervalが正常に算出されないことがあるのでChatGPTに何とかしていただいています（詳細は [#補足](#scrollTo=8VNVk4nsC8LA&line=4&uniqifier=1) を参照）  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/senu3/-/blob/main/utils/Langextract_Colab_ja.ipynb)

In [ ]:
# --- 環境準備 ---
%pip install langextract

In [ ]:
import os
import getpass
import json
import textwrap
import unicodedata, re
import langextract as lx
from langextract import resolver as resolver_lib
from langextract.core import data as core_data
from IPython.display import display, HTML

## APIキーの設定方法
- [3]の実行後、テキスト入力欄が表示されたら自分のAPIキーを入力してください。
- 環境変数があるときは設定不要です（ローカル時）

In [ ]:
# --- APIキー ---
# 環境変数に設定されていなければテキスト入力欄を表示する（ノートに保存しない）
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY (won't be saved to disk): ")

In [ ]:
# --- 分析対象テキスト（実際に抽出したい小説本文） ---
input_text = textwrap.dedent("""
雨が激しく降り注いでいる夜だった。田中は会社から傘を持たずに駅から走って帰ってきた。びしょ濡れになった服を脱ぎ捨てて、熱いシャワーを浴びた。
今日も疲れたなと呟きながら、コーヒーを淹れる。窓の外では雷が鳴り響き、稲妻が空を照らしていた。そんな嵐の夜に、玄関のチャイムが鳴った。
時計を見れば午後11時を回っている。こんな遅い時間に誰だろう？　田中は恐る恐るドアを開けると、見知らぬ女が立っていた。彼女は全身ずぶ濡れで、青白い顔をしている。
「すみません、道に迷ってしまって……」女は震え声で言った。女の顔は幼なじみとよく似ていた。
戸惑う田中をよそに、女は安堵の表情を浮かべて、玄関に足を踏み入れる。その時、田中は奇妙な事に気付いた。女の足跡が床に残っていないのだ。
""")

# コード本体

In [ ]:
# --- 抽出のためのプロンプトを定義 ---
prompt = textwrap.dedent("""
Identify all characters and settings in the text.
For each Character, extract: role, appearance, emotion, background.
For each Setting, extract: location, time_period, place_type.
Use the exact text spans. Do not return empty results.
""")

In [ ]:
# --- Few-shot examples（簡単な注釈例） ---
examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""
早朝に、エルナは朝日の中で長い金髪を揺らしながら、故郷の村の門をじっと見つめていた。
強くなりたいと願い、幼い頃から剣の稽古を重ねている少女だ。
将来は王都で騎士になることを夢見ている。
"""),
        extractions=[
            lx.data.Extraction(
                extraction_class="Character",
                extraction_text="エルナ",
                attributes={
                    "role": "少女",
                    "appearance": "長い金髪",
                    "emotion": "強くなりたい",
                    "background": "幼い頃から剣の稽古を重ねている、将来は王都で騎士になることを夢見ている"
                }
            ),
            lx.data.Extraction(
                extraction_class="Setting",
                extraction_text="村の門",
                attributes={
                    "location": "故郷の村",
                    "time_period": "早朝",
                    "place_type": "門",
                }
            )
        ]
    )
]

In [ ]:
# --- LangExtract による抽出実行 ---
# 抽出前にテキストを正規化してから抽出を行う（全角/半角や空白の差を減らす）

# normalize input_text: NFKC and collapse whitespace
normalized_text = unicodedata.normalize('NFKC', input_text)
normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
try:
    result = lx.extract(
        text_or_documents=normalized_text,
        prompt_description=prompt,
        examples=examples,
        model_id="gemini-2.5-flash",  # 適切なモデル名に変えてください（Gemini や OpenAI 等）
    )
except Exception as e:
    raise

In [ ]:
# --- 抽出直後の検査セル ---
# 各 extraction が char_interval と alignment_status を持っているか確認する（可視化の必須条件）
valid_count = sum(
    1 for e in getattr(result, "extractions", [])
    if getattr(e, "char_interval", None)
    and getattr(e.char_interval, "start_pos", None) is not None
)

In [ ]:
# --- 試行: Resolver.align を使って再アライン ---
# Resolver を作成（出力フォーマットに合わせる）
res = resolver_lib.Resolver(format_type=core_data.FormatType.JSON)
# 低めの閾値で fuzzy alignment を試行
aligned = list(
    res.align(
        result.extractions,
        result.text,
        token_offset=0,
        char_offset=0,
        enable_fuzzy_alignment=True,
        fuzzy_alignment_threshold=0.6,
        accept_match_lesser=True,
    )
)

In [ ]:
# --- ワークアラウンド: 単純なサブストリング照合で char_interval を埋める（必要時のみ実行） ---
# 注意: これはヒューリスティックで、重複や曖昧な一致に注意してください。
doc = {'text': result.text, 'extractions': []}
for e in result.extractions:
    ent = {
        'extraction_class': e.extraction_class,
        'extraction_text': e.extraction_text,
        'attributes': getattr(e, 'attributes', None),
        'char_interval': None,
        'token_interval': None,
        'alignment_status': None,
    }
    txt = e.extraction_text or ''
    if txt:
        idx = result.text.find(txt)
        if idx != -1:
            ent['char_interval'] = {'start_pos': idx, 'end_pos': idx + len(txt)}
            # alignment_status must match AlignmentStatus enum values (lowercase)
            ent['alignment_status'] = 'match_exact'
    doc['extractions'].append(ent)
with open('extraction_results_fixed.jsonl', 'w', encoding='utf-8') as f:
    f.write(json.dumps(doc, ensure_ascii=False) + '\n')
# 最小限の出力: JSONLファイル生成
print('Wrote extraction_results_fixed.jsonl.')

Wrote extraction_results_fixed.jsonl.


In [ ]:
# --- 保存とビジュアライゼーション ---
# 固定済 JSONL を可視化
html_content = lx.visualize('extraction_results_fixed.jsonl')
with open('visualization_fixed.html','w', encoding='utf-8') as hf:
    if hasattr(html_content, 'data'):
        hf.write(html_content.data)
    else:
        hf.write(html_content)
display(HTML(html_content.data if hasattr(html_content,'data') else html_content))

LangExtract: Loading extraction_results_fixed.jsonl: 100%|██████████| 3.02k/3.02k [00:00<00:00, 7.14MB/s]

✓ Loaded 1 documents from extraction_results_fixed.jsonl


# 【補足】抽出→可視化で発生した問題と解決方法

チャッピーからの報告をそのまま載せます。

## 概要
LangExtract を使って小説テキストから Character/Setting を抽出した後、`lx.visualize()` による HTML アニメーション可視化を試みたところ、画面に「No valid extractions to animate.」と表示され、可視化が動きませんでした。原因は、可視化が要求する `char_interval`（抽出の文字位置情報）が全ての extraction で `None` になっていたためです。

## 発生した現象（観察結果）
- ノートブックの検査出力（追加した検査セル）で各 extraction は存在したが `char_interval` が None、`alignment_status` も None。例：
  - [0] class=Character, text=エルナ
    - char_interval: None
    - alignment_status: None
    - attributes: {...}
- `lx.visualize()` の内部では `_filter_valid_extractions()` により `char_interval` が有効でない抽出は除外され、結果的に表示する要素がなくなると "No valid extractions to animate." を返す。
- 手動で原文中を検索すると、`extraction_text` は本文中に見つかった（raw find / norm find が正の値）ため、抽出テキスト自体は本文と一致していた。
- ノートブックで実行した `Resolver.align(...)` も `aligned count` を返したが、`result.extractions` 自体の `char_interval` は更新されていなかった（つまり align の結果が `result` に反映されていなかった可能性が高い）。
- `extraction_results.jsonl` をロードして確認したところ、extractions の各要素に `char_interval: null` が保存されていた。

## 根本原因
- 可視化は `char_interval`（start_pos / end_pos）が存在することを必須条件としている。
- `char_interval` が None なのは、パイプラインのどこか（resolver.align の処理またはその結果を `result` に反映する処理）で文字位置情報が埋められていない／保存されていないため。
- また、ワークアラウンドで作成した JSONL に書き出した `alignment_status` の値が Enum 定義（`AlignmentStatus`）と合致していなかった（大文字の 'MATCH_EXACT' 等）は ValueError を起こすため、enum 値に合わせて小文字（`match_exact` など）に揃える必要があった。

## 取った対応（今回の修正）
1. ノートブックに検査セルを追加して、`result.extractions` の中身（`char_interval`, `alignment_status`, `attributes`）を可視化前に確認できるようにした。
2. `Resolver.align` を低めの閾値（fuzzy_alignment_threshold=0.6）で再実行するセルを追加し、align の動作を確認した。
3. ワークアラウンドとして、`extraction_text` を本文から単純に `find()` して `char_interval` を埋めるセルを追加し、`extraction_results_fixed.jsonl` を生成するようにした。
   - この書き出しで `alignment_status` は `AlignmentStatus` Enum の値に合わせて小文字（例: `match_exact`）で出力するように修正した。
4. 生成した `extraction_results_fixed.jsonl` を `lx.visualize()` に渡して `visualization_fixed.html` を作成し、可視化が動くことを確認できるようにした（ノートブック上で実行する想定）。

## 実行手順（ノートブック上での簡単な流れ）
1. 依存インストール
2. API キーを設定（既に環境変数で設定されている場合は不要）
3. 抽出（既にある `input_text` を使う）
4. 抽出直後に追加した検査セルを実行して `result.extractions` の各フィールドを確認。
5. （必要なら）`Resolver.align` を低閾値で試すセルを実行してアラインを試行。
6. ワークアラウンドセルを実行して `extraction_results_fixed.jsonl` を生成。
7. `lx.visualize('extraction_results_fixed.jsonl')` を実行して生成された HTML を表示／保存（例: `visualization_fixed.html`）。

## 恒久対策（推奨）
- 抽出パイプラインを見直し、`lx.extract()` の内部で align が正常に実行され、戻り値の `AnnotatedDocument` に `char_interval` が正しく埋められることを確認する。
  - `lx.extract(..., debug=True)` でログを出力させ、align の段階で何が起きているか確認する。
  - resolver の `fence_output` や `format_type`（JSON/YAML）のミスマッチがないか確認する。モデルに渡す出力フォーマット例が prompt/例と合っていることを確認する。
- 日本語の表記差（全角/半角、異体字、送り仮名、句読点）への耐性を上げるため、resolver 側で正規化（NFKC）や空白除去などの前処理を実装する。
- align の fuzzy 閾値を適切に調整するか、resolver に対して部分一致（MATCH_LESSER）を受け入れる設定を検討する。

## 補足: ワークアラウンドの注意点
- 単純な substring マッチは簡便だが、同一語が複数箇所に現れる場合や曖昧な一致では誤った位置を埋める危険があります。生成された `extraction_results_fixed.jsonl` は目視確認の上で使うことを推奨します。

---
作成日: 2025-09-21